In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Artificial Intelligence Rasikh Ali - Final Project Viva & Revision (Lab 9 - Lab 13)\n",
    "\n",
    "**Project Title:** Retail Price Prediction using Machine Learning\n",
    "\n",
    "This report consolidates all steps from data acquisition (Lab 9) to final model deployment (Lab 12)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import pickle\n",
    "import matplotlib.pyplot as plt\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.linear_model import LinearRegression\n",
    "from sklearn.metrics import r2_score, accuracy_score, precision_score, recall_score, f1_score\n",
    "from sklearn.naive_bayes import BernoulliNB, GaussianNB, MultinomialNB\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from sklearn.tree import DecisionTreeClassifier\n",
    "from sklearn.neighbors import KNeighborsClassifier\n",
    "\n",
    "DATA_FILE = 'processed_data.csv' \n",
    "TARGET_COLUMN = 'Price'"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Stage 1: Data Acquisition & Pre-processing (Labs 9 & 10)\n",
    "\n",
    "This stage covers data loading, initial exploration, cleaning, and preparation of features for modeling."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"### 1. Data Loading (Lab 9 Start)\")\n",
    "try:\n",
    "    data = pd.read_csv(DATA_FILE)\n",
    "    print(f\"Successfully loaded '{DATA_FILE}'.\")\n",
    "except FileNotFoundError:\n",
    "    print(f\"Error: The file '{DATA_FILE}' was not found. Cannot proceed.\")\n",
    "    # exit()\n",
    "\n",
    "print(\"\\n### 2. Initial Exploration (Lab 10 Start)\")\n",
    "print('Shape:', data.shape)\n",
    "print('\\nData Types:')\n",
    "print(data.dtypes)\n",
    "\n",
    "print('\\n### 3. Checking & Handling Null Values')\n",
    "null_counts = data.isnull().sum()\n",
    "print(null_counts)\n",
    "if null_counts.sum() == 0:\n",
    "    print(\"Result: No missing (null) values found.\")\n",
    "\n",
    "print('\\n### 4. Dropping Columns (Example: Store ID)')\n",
    "# Note: For the final deployed model (lr_model.pkl), all 8 features were likely kept.\n",
    "# This step is kept as a demonstration of feature selection.\n",
    "data_clean = data.drop('Store ID', axis=1, errors='ignore')\n",
    "print('Columns after drop:', data_clean.columns.tolist())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Stage 2: Model Training and Evaluation (Lab 11)\n",
    "\n",
    "This stage demonstrates both the core **Regression** task (Price prediction) and the required **Classification** exercise from Lab 11."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### A. Core Project Model: Linear Regression (Regression Task)\n",
    "The Linear Regression model saved as `lr_model.pkl` was chosen for the final Price prediction and achieved high performance in the original training phase."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"### 1. Training the Final Regression Model (Linear Regression)\")\n",
    "\n",
    "# Prepare data using the 8 features seen by lr_model.pkl\n",
    "X_reg = data.drop(TARGET_COLUMN, axis=1)\n",
    "y_reg = data[TARGET_COLUMN]\n",
    "\n",
    "X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)\n",
    "\n",
    "lr_model = LinearRegression()\n",
    "lr_model.fit(X_train_reg, y_train_reg)\n",
    "y_pred_reg = lr_model.predict(X_test_reg)\n",
    "\n",
    "r2 = r2_score(y_test_reg, y_pred_reg)\n",
    "print(f\"R² Score (Regression Accuracy): {round(r2*100, 2)} %\")\n",
    "\n",
    "# Saving the model (already done as lr_model.pkl)\n",
    "pickle.dump(lr_model, open('lr_model.pkl', 'wb'))   \n",
    "print(\"Model saved as 'lr_model.pkl'.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### B. Lab 11 Exercise: Applying Classification Models\n",
    "To fulfill the Lab 11 requirement, the continuous `Price` is converted into a categorical `Price_Class`."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"### 2. Converting to Classification Target\")\n",
    "bins = data[TARGET_COLUMN].quantile([0.25, 0.50, 0.75]).tolist()\n",
    "bins = [data[TARGET_COLUMN].min()] + bins + [data[TARGET_COLUMN].max()]\n",
    "labels = [0, 1, 2, 3] # Low, Medium-Low, Medium-High, High\n",
    "data['Price_Class'] = pd.cut(data[TARGET_COLUMN], bins=bins, labels=labels, include_lowest=True).astype(int)\n",
    "\n",
    "X_cls = data.drop([TARGET_COLUMN, 'Price_Class'], axis=1)\n",
    "y_cls = data['Price_Class']\n",
    "\n",
    "X_train_cls, X_test_cls, Y_train_cls, Y_test_cls = train_test_split(X_cls, y_cls, test_size=0.3, shuffle=False)\n",
    "print(f\"Classification Target (Price_Class) distribution:\\n{y_cls.value_counts()}\")\n",
    "\n",
    "print(\"\\n### 3. Applying Classifiers\")\n",
    "scores = {}\n",
    "classifiers = {\n",
    "    'Bernoulli': BernoulliNB(), 'Random Forest': RandomForestClassifier(random_state=42),\n",
    "    'Gaussian': GaussianNB(), 'Decision Tree': DecisionTreeClassifier(random_state=42),\n",
    "    'Multinomial': MultinomialNB(), 'KNeighbors': KNeighborsClassifier(n_neighbors=5)\n",
    "}\n",
    "\n",
    "for name, classifier in classifiers.items():\n",
    "    classifier.fit(X_train_cls, Y_train_cls)\n",
    "    Y_pred_cls = classifier.predict(X_test_cls)\n",
    "    \n",
    "    scores[name] = {\n",
    "        'accuracy': accuracy_score(Y_test_cls, Y_pred_cls),\n",
    "        'f1': f1_score(Y_test_cls, Y_pred_cls, average='weighted', zero_division=0)\n",
    "    }\n",
    "    print(f\"- {name}: Accuracy={scores[name]['accuracy']:.4f}, F1-Score={scores[name]['f1']:.4f}\")\n",
    "\n",
    "print(\"\\n### 4. Plotting F1 Scores (Lab 11 Task 6)\")\n",
    "tick_label = list(scores.keys())\n",
    "height = [s['f1'] for s in scores.values()]\n",
    "left = np.arange(len(tick_label))\n",
    "\n",
    "plt.figure(figsize=(10, 6))\n",
    "plt.bar(left, height, tick_label=tick_label, width=0.8)\n",
    "plt.xlabel('Classifiers')\n",
    "plt.ylabel('F1 Scores')\n",
    "plt.title('F1 Scores of Applied Classifiers (Lab 11)')\n",
    "plt.show()\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Stage 3: Flask Introduction and Deployment (Lab 12)\n",
    "\n",
    "The final regression model (`lr_model.pkl`) is deployed using the Flask web framework to create a user-friendly prediction application."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 1. Flask Architecture & Setup\n",
    "\n",
    "The deployment follows the standard Flask structure:\n",
    "- **`app.py`**: Main application logic, handles routes and model prediction.\n",
    "- **`templates/index.html`**: Frontend interface for user input and result display.\n",
    "- **`static/style.css`**: Styling for the application.\n",
    "\n",
    "### 2. `app.py` Logic\n",
    "The Python script handles the web server, model loading, and prediction route."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "app_py_code = \"\"\"\n",
    "from flask import Flask, render_template, request\n",
    "import pickle\n",
    "import numpy as np\n",
    "\n",
    "app = Flask(__name__)\n",
    "model = pickle.load(open('lr_model.pkl', 'rb'))\n",
    "\n",
    "feature_names = [\n",
    "    'Date', 'Store ID', 'Product ID', 'Inventory Level',\n",
    "    'Demand Forecast', 'Weather Condition', 'Holiday/Promotion',\n",
    "    'Competitor Pricing'\n",
    "]\n",
    "\n",
    "@app.route('/')\n",
    "def home():\n",
    "    return render_template('index.html')\n",
    "\n",
    "@app.route('/predict', methods=['POST'])\n",
    "def predict():\n",
    "    try:\n",
    "        input_data = [float(request.form.get(f)) for f in feature_names]\n",
    "        final_input = np.array(input_data).reshape(1, -1)\n",
    "        prediction = model.predict(final_input)[0]\n",
    "        prediction = round(prediction, 2)\n",
    "\n",
    "        return render_template('index.html', \n",
    "                             prediction_text=f\"Predicted Price: ₹{prediction}\",\n",
    "                             success=True)\n",
    "\n",
    "    except Exception as e:\n",
    "        return render_template('index.html', prediction_text=f\"Error: Invalid Input - {str(e)}\")\n",
    "\n",
    "if __name__ == \"__main__\":\n",
    "    app.run(debug=True)\n",
    "\"\"\"\n",
    "print(\"### Code for app.py:\\n\")\n",
    "print(app_py_code)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 3. `templates/index.html` and `static/style.css`\n",
    "The frontend provides a form for the 8 features and displays the output. (The full HTML/CSS content was provided in Lab 12 and is assumed to be in the project structure)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"### Deployment Summary\")\n",
    "print(\"To run the application:\")\n",
    "print(\"1. Ensure 'lr_model.pkl', 'processed_data.csv', app.py, templates/index.html, and static/style.css are in the correct structure.\")\n",
    "print(\"2. Run: 'python app.py' in the terminal.\")\n",
    "print(\"3. Access the app at: http://127.0.0.1:5000\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.x"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}